# English (Svarah): CPU notebook

**Runtime: CPU.** No GPU is needed anywhere in this notebook.

| Cell | What it does | When to run |
|---|---|---|
| 1 | Checks what is on Drive and whether an aligner is running | first |
| 2 | Rebuilds everything and **starts the aligner** (1-2 h, runs in the background) | only if cell 1 shows `textgrids.tar` is tiny/missing |
| 3 | Status of the aligner | whenever you want to check |
| 4 | Alignment checks + feature extraction (mel, pitch, energy) | only when cell 3 says NOT RUNNING and the log ends with ALIGN_DONE_AND_SAVED |

Keep this browser tab open while the aligner runs.

## Cell 1: check Drive (CPU)

In [2]:
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess
D = "/content/drive/MyDrive/tts_project"
for n in ["textgrids.tar", "raw_data_svarah.tar", "svarah_split.csv", "repo_patches.diff", "config_Svarah"]:
    p = f"{D}/{n}"
    print(f"{n:22s}", (f"{os.path.getsize(p)/1e6:.2f} MB" if os.path.isfile(p) else "folder/found") if os.path.exists(p) else "MISSING")
!tar -tf {D}/textgrids.tar | head -3
print("CPU cores here:", os.cpu_count())
print("aligner running here:", subprocess.run(["pgrep", "-f", "mfa train"], capture_output=True).returncode == 0)

Mounted at /content/drive
textgrids.tar          32.14 MB
raw_data_svarah.tar    1019.12 MB
svarah_split.csv       1.96 MB
repo_patches.diff      0.01 MB
config_Svarah          folder/found
TextGrid/
TextGrid/S040/
TextGrid/S040/S040_01436.TextGrid
CPU cores here: 2
aligner running here: False


## Cell 2: start the aligner (CPU)

In [3]:
from google.colab import drive
drive.mount('/content/drive')
import os, re, glob, subprocess
from collections import defaultdict
D = "/content/drive/MyDrive/tts_project"
if subprocess.run(["pgrep", "-f", "mfa train"], capture_output=True).returncode == 0:
    raise RuntimeError("Aligner is already running. Nothing to do.")

!pip install -q g2p_en num2words tgt pyworld pypinyin unidecode inflect librosa pyyaml
import nltk
for r in ["averaged_perceptron_tagger_eng", "averaged_perceptron_tagger", "punkt", "punkt_tab", "cmudict"]:
    nltk.download(r, quiet=True)
from g2p_en import G2p

if not os.path.isdir("/content/FastSpeech2"):
    !git clone -q https://github.com/ming024/FastSpeech2.git /content/FastSpeech2
    !cd /content/FastSpeech2 && git apply {D}/repo_patches.diff && echo "patches applied"
    !cp -r {D}/config_Svarah /content/FastSpeech2/config/Svarah
    !cd /content/FastSpeech2/hifigan && unzip -o -q generator_universal.pth.tar.zip
if not os.path.isdir("/content/raw_data/Svarah"):
    !tar -xf {D}/raw_data_svarah.tar -C /content
fix = lambda t: " ".join(w.strip("'") for w in t.split() if w.strip("'"))
for lab in glob.glob("/content/raw_data/Svarah/*/*.lab"):
    t = open(lab).read(); n = fix(t)
    if n != t: open(lab, "w").write(n)

lex = defaultdict(list)
for l in open("/content/FastSpeech2/lexicon/librispeech-lexicon.txt"):
    p = re.split(r"\s+", l.strip())
    if len(p) > 1 and " ".join(p[1:]) not in lex[p[0].lower()]:
        lex[p[0].lower()].append(" ".join(p[1:]))
g2p = G2p()
LETTER = {"a":"EY1","b":"B IY1","c":"S IY1","d":"D IY1","e":"IY1","f":"EH1 F","g":"JH IY1",
          "h":"EY1 CH","i":"AY1","j":"JH EY1","k":"K EY1","l":"EH1 L","m":"EH1 M","n":"EH1 N",
          "o":"OW1","p":"P IY1","q":"K Y UW1","r":"AA1 R","s":"EH1 S","t":"T IY1","u":"Y UW1",
          "v":"V IY1","w":"D AH1 B AH0 L Y UW0","x":"EH1 K S","y":"W AY1","z":"Z IY1"}
ACRONYMS = {"aicte","esic","epfo","upi","otp","uan","sbi","tv"}
MANUAL = {"mpin": ["EH1 M P IH1 N"]}
words = set()
for lab in glob.glob("/content/raw_data/Svarah/*/*.lab"):
    words.update(open(lab).read().split())
os.makedirs("/content/mfa", exist_ok=True)
with open("/content/mfa/dict.txt", "w") as out:
    for w in sorted(words):
        prons = list(lex.get(w, []))
        if not prons:
            ph = [x for x in g2p(w) if re.fullmatch(r"[A-Z]+[0-2]?", x)]
            if ph: prons.append(" ".join(ph))
        if w in ACRONYMS: prons.append(" ".join(LETTER[c] for c in w))
        prons += MANUAL.get(w, [])
        for p in dict.fromkeys(prons): out.write(f"{w}\t{p}\n")
print("vocab:", len(words), "| dict lines:", sum(1 for _ in open("/content/mfa/dict.txt")))

if not os.path.exists("/content/mfa_env"):
    !curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj -C /content bin/micromamba
    !/content/bin/micromamba create -y -q -p /content/mfa_env -c conda-forge python=3.11 montreal-forced-aligner
os.makedirs("/content/mfa_logs", exist_ok=True)
os.makedirs("/content/FastSpeech2/preprocessed_data/Svarah", exist_ok=True)
nj = os.cpu_count() or 2
cmd = (f"MFA_ROOT_DIR=/content/mfa_root /content/bin/micromamba run -p /content/mfa_env "
       f"mfa train --clean -j {nj} --output_directory /content/FastSpeech2/preprocessed_data/Svarah/TextGrid "
       f"/content/raw_data/Svarah /content/mfa/dict.txt /content/mfa/svarah_acoustic.zip "
       f"> /content/mfa_logs/train.log 2>&1 "
       f"&& cd /content/FastSpeech2/preprocessed_data/Svarah && tar -cf {D}/textgrids.tar TextGrid "
       f"&& cp /content/mfa/svarah_acoustic.zip {D}/ && echo ALIGN_DONE_AND_SAVED >> /content/mfa_logs/train.log")
subprocess.Popen(["bash", "-c", cmd], start_new_session=True)
print(f"aligner started with {nj} CPU jobs")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
patches applied
vocab: 6503 | dict lines: 7666
aligner started with 2 CPU jobs


## Cell 3: aligner status (CPU)

In [4]:
import subprocess
running = subprocess.run(["pgrep", "-f", "mfa train"], capture_output=True).returncode == 0
b = subprocess.run("tr '\\r' '\\n' < /content/mfa_logs/train.log | grep -a -E 'INFO|ERROR|ALIGN_DONE' | tail -3", shell=True, capture_output=True).stdout
print("ALIGNER:", "RUNNING" if running else "NOT RUNNING"); print(b.decode("utf-8", errors="replace"))

ALIGNER: RUNNING
 INFO     Initializing multiprocessing jobs...                                  
 INFO     Normalizing text...                                                   
 INFO     Generating MFCCs...                                                   



## Cell 4: alignment checks + features (CPU). Run only after the aligner has finished

In [5]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys, glob, subprocess, collections, numpy as np, pandas as pd
D = "/content/drive/MyDrive/tts_project"

# 1) what is on Drive?
for n in ["textgrids.tar", "raw_data_svarah.tar", "svarah_split.csv", "repo_patches.diff", "config_Svarah"]:
    print(f"{n:22s}", "found" if os.path.exists(f"{D}/{n}") else "MISSING")
tgp = f"{D}/textgrids.tar"
if not os.path.exists(tgp) or os.path.getsize(tgp) < 1_000_000:
    raise SystemExit("textgrids.tar is missing or empty: the English alignment has not finished. Check the status cell first.")

# 2) restore repo, corpus, TextGrids
!pip install -q pyworld pypinyin unidecode inflect librosa pyyaml tgt scikit-learn
if not os.path.isdir("/content/FastSpeech2"):
    !git clone -q https://github.com/ming024/FastSpeech2.git /content/FastSpeech2
    !cd /content/FastSpeech2 && git apply {D}/repo_patches.diff && echo "repo patches applied"
    !cp -r {D}/config_Svarah /content/FastSpeech2/config/Svarah
    !cd /content/FastSpeech2/hifigan && unzip -o -q generator_universal.pth.tar.zip
if not os.path.isdir("/content/raw_data/Svarah"):
    !tar -xf {D}/raw_data_svarah.tar -C /content
P = "/content/FastSpeech2/preprocessed_data/Svarah"
os.makedirs(P, exist_ok=True)
if not os.path.isdir(f"{P}/TextGrid"):
    !tar -xf {D}/textgrids.tar -C {P}
f = pd.read_csv(f"{D}/svarah_split.csv")
open("/content/FastSpeech2/split_val_utts.txt", "w").write("\n".join(f.utt[f.split == "val"]))

# 3) alignment sanity checks
import tgt
sys.path.insert(0, "/content/FastSpeech2")
from text.symbols import symbols
tgs = sorted(glob.glob(f"{P}/TextGrid/*/*.TextGrid"))
print("TextGrids:", len(tgs), "of", len(f), "utterances")
labels, durs = collections.Counter(), []
for p in tgs:
    for iv in tgt.io.read_textgrid(p).get_tier_by_name("phones"):
        labels[iv.text] += 1; durs.append(iv.end_time - iv.start_time)
unknown = {k: v for k, v in labels.items() if k and ("@" + k) not in symbols}
d = np.array(durs)
print("distinct phone labels:", len(labels), "| empty (silence) labels:", labels.get("", 0), "| labels unknown to the model:", unknown)
print("phone duration (s): median %.3f | p5 %.3f | p95 %.3f | max %.2f" % (np.median(d), *np.percentile(d, [5, 95]), d.max()))
tg = tgt.io.read_textgrid(tgs[0]); print(tgs[0])
print(" ".join(f"{iv.text or 'sil'}:{(iv.end_time-iv.start_time)*1000:.0f}" for iv in tg.get_tier_by_name("phones")))
print(" ".join(iv.text for iv in tg.get_tier_by_name("words")))
if unknown or len(tgs) < 0.95 * len(f):
    raise SystemExit("Alignment check failed (see above). Paste this output before preprocessing.")

# 4) feature extraction (mel, pitch, energy) with our own train/val split
os.chdir("/content/FastSpeech2")
!python preprocess.py config/Svarah/preprocess.yaml 2>&1 | tail -15
for n in ["train.txt", "val.txt"]:
    print(n, sum(1 for _ in open(f"{P}/{n}", encoding="utf-8")), "lines")
print("speakers:", len(__import__("json").load(open(f"{P}/speakers.json"))), "| stats:", open(f"{P}/stats.json").read())
!head -2 {P}/train.txt
!cd /content/FastSpeech2/preprocessed_data && tar -cf {D}/preprocessed_Svarah.tar Svarah && ls -lh {D}/preprocessed_Svarah.tar


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
textgrids.tar          found
raw_data_svarah.tar    found
svarah_split.csv       found
repo_patches.diff      found
config_Svarah          found
TextGrids: 3992 of 4048 utterances
distinct phone labels: 69 | empty (silence) labels: 0 | labels unknown to the model: {}
phone duration (s): median 0.070 | p5 0.030 | p95 0.180 | max 4.91
/content/FastSpeech2/preprocessed_data/Svarah/TextGrid/S000/S000_00014.TextGrid
S:250 IY1:30 DH:30 IH0:30 S:30 T:180 R:10 AE1:120 K:140 AE1:60 Z:50 M:80 AY1:90 P:80 R:10 AH0:70 F:30 EH1:30 SH:110 AH0:30 N:30 AH0:50 L:60 IH2:60 Z:80 AH0:70 M:130 B:220 AH1:80 T:140 AY1:120 Y:60 UW1:30 Z:90 D:30 T:60 AH0:30 M:110 AE1:60 N:60 AH0:40 JH:50 S:150 AH1:40 M:70 T:120 AY1:150 M:40 IH0:30 N:140 R:60 AY1:120 T:110 IH0:40 NG:160 EH1:60 Z:60 M:40 AY1:30
see this track as my professionalism but i used to manage some time in writing as my
/usr/lo

In [6]:
import glob, tgt
P = "/content/FastSpeech2/preprocessed_data/Svarah"
n_bad, n = 0, 0
for p in sorted(glob.glob(f"{P}/TextGrid/*/*.TextGrid"))[:300]:
    iv = list(tgt.io.read_textgrid(p).get_tier_by_name("phones"))
    gaps = sum(max(0, b.start_time - a.end_time) for a, b in zip(iv, iv[1:]))
    n += 1; n_bad += gaps > 0.02
print(f"{n_bad} of {n} utterances have gaps inside the phone tier")

258 of 300 utterances have gaps inside the phone tier


In [7]:
import subprocess
pids = subprocess.run(["pgrep", "-f", "mfa train"], capture_output=True, text=True).stdout.split()
print("ALIGNER:", "RUNNING" if pids else "NOT RUNNING")
if pids:
    print("running for (d-hh:mm:ss):", subprocess.run(["ps", "-o", "etime=", "-p", pids[0]], capture_output=True, text=True).stdout.strip())
raw = subprocess.run("tr '\\r' '\\n' < /content/mfa_logs/train.log", shell=True, capture_output=True).stdout.decode("utf-8", "replace")
lines = [l.strip() for l in raw.splitlines() if any(k in l for k in ("INFO", "ERROR", "Error", "ALIGN_DONE"))]
its = [l for l in lines if "Iteration" in l]
print("last stage line:", its[-1] if its else "(none yet)")
print("\n".join(lines[-4:]))

ALIGNER: RUNNING
running for (d-hh:mm:ss): 27:24
last stage line: INFO     monophone - Iteration 9 of 40
INFO     Generating alignments...
INFO     Accumulating statistics...
INFO     monophone - Iteration 9 of 40
INFO     Generating alignments...


In [8]:
import os
os.chdir("/content/FastSpeech2")
!cat synthesize.py
print("=========== model/fastspeech2.py ===========")
!cat model/fastspeech2.py
print("=========== model/modules.py (first 190 lines) ===========")
!sed -n '1,190p' model/modules.py

import re
import argparse
from string import punctuation

import torch
import yaml
import numpy as np
from torch.utils.data import DataLoader
from g2p_en import G2p
from pypinyin import pinyin, Style

from utils.model import get_model, get_vocoder
from utils.tools import to_device, synth_samples
from dataset import TextDataset
from text import text_to_sequence

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def read_lexicon(lex_path):
    lexicon = {}
    with open(lex_path) as f:
        for line in f:
            temp = re.split(r"\s+", line.strip("\n"))
            word = temp[0]
            phones = temp[1:]
            if word.lower() not in lexicon:
                lexicon[word.lower()] = phones
    return lexicon


def preprocess_english(text, preprocess_config):
    text = text.rstrip(punctuation)
    lexicon = read_lexicon(preprocess_config["path"]["lexicon_path"])

    g2p = G2p()
    phones = []
    words = re.split(r"([,;.\-\?\!\s+])", text)
    for

In [9]:
from google.colab import drive
drive.mount('/content/drive')
import os
D = "/content/drive/MyDrive/tts_project"
for n in ["textgrids.tar", "textgrids_telugu.tar", "preprocessed_Svarah.tar", "preprocessed_Telugu.tar"]:
    p = f"{D}/{n}"
    print(f"{n:26s}", f"{os.path.getsize(p)/1e6:.2f} MB" if os.path.exists(p) else "missing")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
textgrids.tar              32.14 MB
textgrids_telugu.tar       30.00 MB
preprocessed_Svarah.tar    495.90 MB
preprocessed_Telugu.tar    464.61 MB


In [10]:
import subprocess
pids = subprocess.run(["pgrep", "-f", "mfa train"], capture_output=True, text=True).stdout.split()
print("ALIGNER:", "RUNNING" if pids else "NOT RUNNING")
if pids:
    print("running for ([dd-][hh:]mm:ss):", subprocess.run(["ps", "-o", "etime=", "-p", pids[0]], capture_output=True, text=True).stdout.strip())
raw = subprocess.run("tr '\\r' '\\n' < /content/mfa_logs/train.log", shell=True, capture_output=True).stdout.decode("utf-8", "replace")
lines = [l.strip() for l in raw.splitlines() if any(k in l for k in ("INFO", "ERROR", "Error", "ALIGN_DONE"))]
its = [l for l in lines if "Iteration" in l]
print("last stage line:", its[-1] if its else "(none yet)")
print("\n".join(lines[-3:]))

ALIGNER: RUNNING
running for ([dd-][hh:]mm:ss): 27:39
last stage line: INFO     monophone - Iteration 9 of 40
INFO     Accumulating statistics...
INFO     monophone - Iteration 9 of 40
INFO     Generating alignments...


In [11]:
import subprocess, time
subprocess.run(["pkill", "-f", "mfa train"]); subprocess.run(["pkill", "-f", "micromamba"])
time.sleep(3)
print("aligner still running:", subprocess.run(["pgrep", "-f", "mfa train"], capture_output=True).returncode == 0)

aligner still running: False


In [12]:
import os, time, shutil, subprocess
D = "/content/drive/MyDrive/tts_project"
LOG = "/content/mfa_logs/train.log"
def running(): return subprocess.run(["pgrep", "-f", "mfa train"], capture_output=True).returncode == 0
def log_tail(n=1500):
    b = subprocess.run(f"tail -c {n} {LOG}", shell=True, capture_output=True).stdout
    return b.decode("utf-8", errors="replace").replace("\r", "\n")
def last_line():
    lines = [l.strip() for l in log_tail().splitlines() if l.strip()]
    return lines[-1][:150] if lines else "(no log yet)"

missing = [p for p in ["/content/mfa/dict.txt", "/content/raw_data/Svarah", "/content/mfa_env", "/content/bin/micromamba"] if not os.path.exists(p)]
if missing: raise SystemExit(f"Missing {missing}: the runtime was reset. Run the full restart cell first (the long one that rebuilds everything).")
if running(): raise SystemExit("An aligner is already running. Nothing to do.")

open("/content/mfa_fast.yaml", "w").write("""beam: 10
retry_beam: 40
training:
  - monophone:
      num_iterations: 30
      max_gaussians: 1000
      boost_silence: 1.25
  - triphone:
      num_iterations: 25
      num_leaves: 2000
      max_gaussians: 10000
      cluster_threshold: -1
      boost_silence: 1.25
      power: 0.25
""")
shutil.rmtree("/content/FastSpeech2/preprocessed_data/Svarah/TextGrid", ignore_errors=True)
os.makedirs("/content/mfa_logs", exist_ok=True)
nj = os.cpu_count() or 2
cmd = (f"MFA_ROOT_DIR=/content/mfa_root /content/bin/micromamba run -p /content/mfa_env "
       f"mfa train --clean -j {nj} --config_path /content/mfa_fast.yaml "
       f"--output_directory /content/FastSpeech2/preprocessed_data/Svarah/TextGrid "
       f"/content/raw_data/Svarah /content/mfa/dict.txt /content/mfa/svarah_acoustic.zip > {LOG} 2>&1 "
       f"&& cd /content/FastSpeech2/preprocessed_data/Svarah && tar -cf {D}/textgrids.tar TextGrid "
       f"&& cp /content/mfa/svarah_acoustic.zip {D}/ && echo ALIGN_DONE_AND_SAVED >> {LOG}")
subprocess.Popen(["bash", "-c", cmd], start_new_session=True)
time.sleep(60)
if not running() and "ALIGN_DONE_AND_SAVED" not in log_tail(20000):
    raise SystemExit("The aligner exited within a minute. Paste this:\n" + log_tail(3000))
print("started OK |", last_line())

t0 = time.time()
while running() and time.time() - t0 < 5 * 3600:
    time.sleep(120)
    print(f"[{(time.time()-t0)/60:.0f} min]", last_line())
print("FINISHED. Saved to Drive:", "ALIGN_DONE_AND_SAVED" in log_tail(20000))

started OK | INFO     Generating initial alignments...
[2 min] INFO     Generating alignments...
[4 min] INFO     Generating alignments...
[6 min] INFO     Generating alignments...
[8 min] INFO     Accumulating statistics...
[10 min] INFO     Generating alignments...
[12 min] INFO     Accumulating statistics...
[14 min] INFO     Accumulating statistics...
[16 min] INFO     Compiling training graphs...
[18 min] INFO     Analyzing alignment quality...
[20 min] INFO     Accumulating statistics...
[22 min] INFO     Generating alignments...
[24 min] INFO     Accumulating statistics...
[26 min] INFO     Accumulating statistics...
[28 min] INFO     Generating alignments...
[30 min] INFO     Analyzing alignment quality...
[32 min] ALIGN_DONE_AND_SAVED
FINISHED. Saved to Drive: True
